In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder, label_binarize
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, VotingClassifier
from sklearn.feature_selection import RFE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, auc, mean_absolute_error, mean_squared_error, r2_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import joblib

In [ ]:
search_paths = glob.glob("/kaggle/input/**/krishijal_master_dataset.csv", recursive=True) + glob.glob("**/krishijal_master_dataset.csv", recursive=True) + ["krishijal_master_dataset.csv"]
dataset_path = search_paths[0]
for path_candidate in search_paths:
    candidate_df = pd.read_csv(path_candidate, usecols=["granularity"])
    if "district_baseline" in candidate_df["granularity"].values:
        dataset_path = path_candidate
        break
df = pd.read_csv(dataset_path)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset=["temperature", "rainfall", "humidity", "effective_soil_wetness", "crop_type", "growth_stage"])
df = df.drop_duplicates()

In [ ]:
selected_features = [
    "temperature", "temp_max", "temp_min", "rainfall", "humidity",
    "effective_soil_wetness", "solar_radiation", "soil_ph", "soil_organic_carbon",
    "ndvi", "et0", "etc", "crop_type", "growth_stage"
]
df_train = df[df["granularity"].isin(["daily", "monthly"])].copy()
X = df_train[selected_features].copy()
y_class = df_train["stress_level_code"].copy()
y_reg = df_train["irrigation_priority_index"].copy()

class_counts = df_train["stress_level"].value_counts().reindex(["Low", "Moderate", "High", "Severe"])
class_pct = (df_train["stress_level"].value_counts(normalize=True).reindex(["Low", "Moderate", "High", "Severe"]) * 100).round(2)
imbalance_summary = pd.DataFrame({"Count": class_counts, "Percentage": class_pct})
print("Exact duplicate rows:", df_train.duplicated().sum())
print("Duplicate feature rows:", X.duplicated().sum())
print("Leakage columns excluded from X:", ["cwsi", "stress_coefficient_ks", "eta", "water_deficit", "irrigation_priority_index", "stress_level"])
imbalance_summary

In [ ]:
crop_encoder = LabelEncoder()
X["crop_code"] = crop_encoder.fit_transform(X["crop_type"])

stage_encoder = LabelEncoder()
X["stage_code"] = stage_encoder.fit_transform(X["growth_stage"])

feature_cols = [
    "temperature", "temp_max", "temp_min", "rainfall", "humidity",
    "effective_soil_wetness", "solar_radiation", "soil_ph", "soil_organic_carbon",
    "ndvi", "et0", "etc", "crop_code", "stage_code"
]
X = X[feature_cols]

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(x=df_train["stress_level"], hue=df_train["stress_level"], order=["Low", "Moderate", "High", "Severe"], palette="viridis", legend=False)
plt.title("Distribution of Crop Stress Severity")
plt.xlabel("Stress Level")
plt.ylabel("Number of Observations")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df_train["irrigation_priority_index"], bins=25, kde=True, color="teal")
plt.title("Distribution of Irrigation Priority Index")
plt.xlabel("Irrigation Priority Index (0 to 100)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
corr_matrix = X.corr()
high_corr_pairs = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        if abs(corr_matrix.iloc[i, j]) > 0.85:
            high_corr_pairs.append((feature_cols[i], feature_cols[j], round(corr_matrix.iloc[i, j], 4)))
print("Highly correlated feature pairs (|r| > 0.85):", high_corr_pairs)
print("Target correlation ranking:")
print(X.corrwith(y_class).sort_values(key=abs, ascending=False).round(4))

plt.figure(figsize=(9, 6))
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(x=df_train["stress_level"], y=df_train["temperature"], hue=df_train["stress_level"], order=["Low", "Moderate", "High", "Severe"], palette="Set2", legend=False)
plt.title("Temperature by Crop Stress Severity")
plt.xlabel("Stress Level")
plt.ylabel("Temperature (°C)")
plt.show()

In [ ]:
X_train, X_test, y_train_class, y_test_class, y_train_reg, y_test_reg = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

prep_rows = []
scaler_options = [
    ("StandardScaler", StandardScaler(), None),
    ("MinMaxScaler", MinMaxScaler(), None),
    ("RobustScaler", RobustScaler(), None),
    ("StandardScaler + PCA(8)", StandardScaler(), PCA(n_components=8, random_state=42))
]
probe_models = [
    ("Logistic Regression", LogisticRegression(max_iter=2000, random_state=42)),
    ("KNN (k=5)", KNeighborsClassifier(n_neighbors=5)),
    ("Random Forest", RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42))
]
for s_name, s_obj, dim_red in scaler_options:
    for m_name, m_obj in probe_models:
        pipe_steps = [("scaler", s_obj)]
        if dim_red is not None:
            pipe_steps.append(("pca", dim_red))
        pipe_steps.append(("clf", m_obj))
        probe_pipe = ImbPipeline(pipe_steps)
        cv_out = cross_validate(probe_pipe, X_train, y_train_class, cv=cv_strategy, scoring=["accuracy", "f1_weighted", "f1_macro"], n_jobs=-1)
        prep_rows.append({
            "Preprocessing": s_name,
            "Model": m_name,
            "CV Accuracy": round(cv_out["test_accuracy"].mean(), 4),
            "CV Weighted F1": round(cv_out["test_f1_weighted"].mean(), 4),
            "CV Macro F1": round(cv_out["test_f1_macro"].mean(), 4)
        })
preprocessing_comparison = pd.DataFrame(prep_rows)
preprocessing_comparison

In [ ]:
rf_probe = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
rf_probe.fit(X_train_scaled, y_train_class)
feature_importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": np.round(rf_probe.feature_importances_, 4)
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

rfe_rows = []
for n_keep in [5, 8, 12, 14]:
    if n_keep < 14:
        rfe_pipe = ImbPipeline([
            ("scaler", StandardScaler()),
            ("rfe", RFE(RandomForestClassifier(n_estimators=50, max_depth=12, random_state=42), n_features_to_select=n_keep)),
            ("clf", RandomForestClassifier(n_estimators=150, max_depth=16, random_state=42))
        ])
    else:
        rfe_pipe = ImbPipeline([
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(n_estimators=150, max_depth=16, random_state=42))
        ])
    cv_rfe = cross_validate(rfe_pipe, X_train, y_train_class, cv=cv_strategy, scoring=["accuracy", "f1_weighted", "f1_macro"], n_jobs=-1)
    rfe_rows.append({
        "Selected Features": n_keep,
        "CV Accuracy": round(cv_rfe["test_accuracy"].mean(), 4),
        "CV Weighted F1": round(cv_rfe["test_f1_weighted"].mean(), 4),
        "CV Macro F1": round(cv_rfe["test_f1_macro"].mean(), 4)
    })
print("Feature Importance Ranking:")
print(feature_importance_df)
pd.DataFrame(rfe_rows)

In [ ]:
imbalance_rows = []
resampling_configs = [
    ("No Resampling", None, None),
    ("SMOTE", SMOTE(random_state=42), None),
    ("RandomOverSampler", RandomOverSampler(random_state=42), None),
    ("RandomUnderSampler", RandomUnderSampler(random_state=42), None),
    ("Class Weights (balanced)", None, "balanced")
]
for r_name, r_obj, cw_val in resampling_configs:
    for m_name, base_clf in [
        ("Logistic Regression", LogisticRegression(max_iter=2000, random_state=42, class_weight=cw_val)),
        ("Decision Tree", DecisionTreeClassifier(max_depth=8, random_state=42, class_weight=cw_val)),
        ("Random Forest", RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, class_weight=cw_val))
    ]:
        steps = [("scaler", StandardScaler())]
        if r_obj is not None:
            steps.append(("sampler", r_obj))
        steps.append(("clf", base_clf))
        imb_pipe = ImbPipeline(steps)
        cv_imb = cross_validate(imb_pipe, X_train, y_train_class, cv=cv_strategy, scoring=["accuracy", "precision_weighted", "recall_weighted", "f1_weighted", "f1_macro"], n_jobs=-1)
        imbalance_rows.append({
            "Resampling Method": r_name,
            "Algorithm": m_name,
            "CV Accuracy": round(cv_imb["test_accuracy"].mean(), 4),
            "CV Precision": round(cv_imb["test_precision_weighted"].mean(), 4),
            "CV Recall": round(cv_imb["test_recall_weighted"].mean(), 4),
            "CV Weighted F1": round(cv_imb["test_f1_weighted"].mean(), 4),
            "CV Macro F1": round(cv_imb["test_f1_macro"].mean(), 4)
        })
imbalance_comparison = pd.DataFrame(imbalance_rows)
imbalance_comparison

In [ ]:
model_logreg = LogisticRegression(max_iter=2000, random_state=42)
model_logreg.fit(X_train_scaled, y_train_class)

model_knn_class = KNeighborsClassifier(n_neighbors=5)
model_knn_class.fit(X_train_scaled, y_train_class)

model_gnb_class = GaussianNB()
model_gnb_class.fit(X_train_scaled, y_train_class)

model_dt_class = DecisionTreeClassifier(max_depth=8, random_state=42)
model_dt_class.fit(X_train_scaled, y_train_class)

model_rf_class = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
model_rf_class.fit(X_train_scaled, y_train_class)

In [ ]:
pred_logreg = model_logreg.predict(X_test_scaled)
pred_knn_class = model_knn_class.predict(X_test_scaled)
pred_gnb_class = model_gnb_class.predict(X_test_scaled)
pred_dt_class = model_dt_class.predict(X_test_scaled)
pred_rf_class = model_rf_class.predict(X_test_scaled)

classification_comparison = pd.DataFrame({
    "Algorithm": [
        "Logistic Regression",
        "K-Nearest Neighbors",
        "Gaussian Naive Bayes",
        "Decision Tree Classifier",
        "Random Forest Classifier"
    ],
    "Accuracy": [
        accuracy_score(y_test_class, pred_logreg),
        accuracy_score(y_test_class, pred_knn_class),
        accuracy_score(y_test_class, pred_gnb_class),
        accuracy_score(y_test_class, pred_dt_class),
        accuracy_score(y_test_class, pred_rf_class)
    ],
    "Precision": [
        precision_score(y_test_class, pred_logreg, average="weighted", zero_division=0),
        precision_score(y_test_class, pred_knn_class, average="weighted", zero_division=0),
        precision_score(y_test_class, pred_gnb_class, average="weighted", zero_division=0),
        precision_score(y_test_class, pred_dt_class, average="weighted", zero_division=0),
        precision_score(y_test_class, pred_rf_class, average="weighted", zero_division=0)
    ],
    "Recall": [
        recall_score(y_test_class, pred_logreg, average="weighted", zero_division=0),
        recall_score(y_test_class, pred_knn_class, average="weighted", zero_division=0),
        recall_score(y_test_class, pred_gnb_class, average="weighted", zero_division=0),
        recall_score(y_test_class, pred_dt_class, average="weighted", zero_division=0),
        recall_score(y_test_class, pred_rf_class, average="weighted", zero_division=0)
    ],
    "Weighted F1-Score": [
        f1_score(y_test_class, pred_logreg, average="weighted", zero_division=0),
        f1_score(y_test_class, pred_knn_class, average="weighted", zero_division=0),
        f1_score(y_test_class, pred_gnb_class, average="weighted", zero_division=0),
        f1_score(y_test_class, pred_dt_class, average="weighted", zero_division=0),
        f1_score(y_test_class, pred_rf_class, average="weighted", zero_division=0)
    ],
    "Macro F1-Score": [
        f1_score(y_test_class, pred_logreg, average="macro", zero_division=0),
        f1_score(y_test_class, pred_knn_class, average="macro", zero_division=0),
        f1_score(y_test_class, pred_gnb_class, average="macro", zero_division=0),
        f1_score(y_test_class, pred_dt_class, average="macro", zero_division=0),
        f1_score(y_test_class, pred_rf_class, average="macro", zero_division=0)
    ]
})
classification_comparison

In [ ]:
model_linreg = LinearRegression()
model_linreg.fit(X_train_scaled, y_train_reg)

model_dt_reg = DecisionTreeRegressor(max_depth=8, random_state=42)
model_dt_reg.fit(X_train_scaled, y_train_reg)

model_rf_reg = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42)
model_rf_reg.fit(X_train_scaled, y_train_reg)

In [ ]:
pred_linreg = model_linreg.predict(X_test_scaled)
pred_dt_reg = model_dt_reg.predict(X_test_scaled)
pred_rf_reg = model_rf_reg.predict(X_test_scaled)

regression_comparison = pd.DataFrame({
    "Algorithm": ["Linear Regression", "Decision Tree Regressor", "Random Forest Regressor"],
    "MAE": [
        mean_absolute_error(y_test_reg, pred_linreg),
        mean_absolute_error(y_test_reg, pred_dt_reg),
        mean_absolute_error(y_test_reg, pred_rf_reg)
    ],
    "MSE": [
        mean_squared_error(y_test_reg, pred_linreg),
        mean_squared_error(y_test_reg, pred_dt_reg),
        mean_squared_error(y_test_reg, pred_rf_reg)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test_reg, pred_linreg)),
        np.sqrt(mean_squared_error(y_test_reg, pred_dt_reg)),
        np.sqrt(mean_squared_error(y_test_reg, pred_rf_reg))
    ],
    "R2 Score": [
        r2_score(y_test_reg, pred_linreg),
        r2_score(y_test_reg, pred_dt_reg),
        r2_score(y_test_reg, pred_rf_reg)
    ]
})
regression_comparison

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test_reg, pred_rf_reg, alpha=0.4, color="forestgreen", s=20)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], color="red", linestyle="--")
plt.title("Actual vs Predicted Irrigation Priority Index")
plt.xlabel("Actual Priority Index")
plt.ylabel("Predicted Priority Index")
plt.show()

In [ ]:
search_lr = RandomizedSearchCV(
    estimator=ImbPipeline([("scaler", StandardScaler()), ("smote", SMOTE(random_state=42)), ("clf", LogisticRegression(max_iter=2000, random_state=42))]),
    param_distributions={"smote__k_neighbors": [3, 5], "clf__C": [1.0, 10.0, 50.0, 100.0], "clf__class_weight": [None, "balanced"]},
    n_iter=10, cv=cv_strategy, scoring="f1_macro", random_state=42, n_jobs=-1
)
search_lr.fit(X_train, y_train_class)

search_knn = RandomizedSearchCV(
    estimator=ImbPipeline([("scaler", MinMaxScaler()), ("smote", SMOTE(random_state=42)), ("clf", KNeighborsClassifier())]),
    param_distributions={"smote__k_neighbors": [3, 5], "clf__n_neighbors": [3, 5, 7], "clf__weights": ["uniform", "distance"], "clf__p": [1, 2]},
    n_iter=10, cv=cv_strategy, scoring="f1_macro", random_state=42, n_jobs=-1
)
search_knn.fit(X_train, y_train_class)

search_dt = RandomizedSearchCV(
    estimator=ImbPipeline([("scaler", StandardScaler()), ("smote", SMOTE(random_state=42)), ("clf", DecisionTreeClassifier(random_state=42))]),
    param_distributions={"smote__k_neighbors": [3, 5], "clf__criterion": ["gini", "entropy"], "clf__max_depth": [8, 12, 16, None], "clf__min_samples_split": [2, 4, 6], "clf__min_samples_leaf": [1, 2], "clf__class_weight": [None, "balanced"]},
    n_iter=15, cv=cv_strategy, scoring="f1_macro", random_state=42, n_jobs=-1
)
search_dt.fit(X_train, y_train_class)

search_rf_tuned = RandomizedSearchCV(
    estimator=ImbPipeline([("scaler", StandardScaler()), ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))]),
    param_distributions={
        "clf__n_estimators": [100, 200, 300],
        "clf__max_depth": [12, 16, 20, None],
        "clf__min_samples_split": [2, 3, 5],
        "clf__min_samples_leaf": [1, 2],
        "clf__max_features": ["sqrt", "log2", None],
        "clf__criterion": ["gini", "entropy"],
        "clf__max_samples": [0.85, 1.0],
        "clf__class_weight": [None, "balanced", "balanced_subsample"]
    },
    n_iter=20, cv=cv_strategy, scoring="f1_macro", random_state=42, n_jobs=-1
)
search_rf_tuned.fit(X_train, y_train_class)

search_rf_smote = RandomizedSearchCV(
    estimator=ImbPipeline([("scaler", StandardScaler()), ("smote", SMOTE(random_state=42)), ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))]),
    param_distributions={
        "smote__k_neighbors": [3, 5],
        "smote__sampling_strategy": ["not majority", "all"],
        "clf__n_estimators": [100, 200, 300],
        "clf__max_depth": [12, 16, 20],
        "clf__min_samples_split": [2, 3, 5],
        "clf__min_samples_leaf": [1, 2],
        "clf__max_features": ["sqrt", "log2"],
        "clf__criterion": ["gini", "entropy"],
        "clf__max_samples": [0.85, 1.0],
        "clf__class_weight": [None, "balanced_subsample"]
    },
    n_iter=20, cv=cv_strategy, scoring="f1_macro", random_state=42, n_jobs=-1
)
search_rf_smote.fit(X_train, y_train_class)

search_rf_smote_rfe = RandomizedSearchCV(
    estimator=ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("rfe", RFE(RandomForestClassifier(n_estimators=50, max_depth=12, random_state=42))),
        ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))
    ]),
    param_distributions={
        "smote__k_neighbors": [3, 5],
        "smote__sampling_strategy": ["not majority", "all"],
        "rfe__n_features_to_select": [5, 8, 10],
        "clf__n_estimators": [150, 200, 300],
        "clf__max_depth": [12, 16, 20],
        "clf__min_samples_split": [2, 4],
        "clf__min_samples_leaf": [1, 2],
        "clf__max_features": ["sqrt", "log2"],
        "clf__criterion": ["gini", "entropy"],
        "clf__class_weight": [None, "balanced_subsample"]
    },
    n_iter=20, cv=cv_strategy, scoring="f1_macro", random_state=42, n_jobs=-1
)
search_rf_smote_rfe.fit(X_train, y_train_class)

cv_search_summary = pd.DataFrame([
    {"Pipeline": "Tuned Logistic Regression + SMOTE", "Best CV Macro F1": round(search_lr.best_score_, 4), "Best Parameters": str(search_lr.best_params_)},
    {"Pipeline": "Tuned KNN + MinMax + SMOTE", "Best CV Macro F1": round(search_knn.best_score_, 4), "Best Parameters": str(search_knn.best_params_)},
    {"Pipeline": "Tuned Decision Tree + SMOTE", "Best CV Macro F1": round(search_dt.best_score_, 4), "Best Parameters": str(search_dt.best_params_)},
    {"Pipeline": "Tuned Random Forest", "Best CV Macro F1": round(search_rf_tuned.best_score_, 4), "Best Parameters": str(search_rf_tuned.best_params_)},
    {"Pipeline": "Best SMOTE Random Forest", "Best CV Macro F1": round(search_rf_smote.best_score_, 4), "Best Parameters": str(search_rf_smote.best_params_)},
    {"Pipeline": "SMOTE + RFE + Tuned Random Forest", "Best CV Macro F1": round(search_rf_smote_rfe.best_score_, 4), "Best Parameters": str(search_rf_smote_rfe.best_params_)}
])
cv_search_summary

In [ ]:
voting_candidate = VotingClassifier(
    estimators=[("smote_rf", search_rf_smote.best_estimator_), ("smote_rfe_rf", search_rf_smote_rfe.best_estimator_)],
    voting="soft"
)

candidate_pipelines = [
    ("Tuned Random Forest", search_rf_tuned.best_estimator_),
    ("Best SMOTE Model", search_rf_smote.best_estimator_),
    ("SMOTE + RFE + Tuned RF", search_rf_smote_rfe.best_estimator_),
    ("Voting Ensemble (SMOTE + RFE)", voting_candidate)
]

selection_rows = []
best_cv_macro = -1.0
best_overall_name = ""
best_model = None
for c_name, c_pipe in candidate_pipelines:
    c_scores = cross_validate(c_pipe, X_train, y_train_class, cv=cv_strategy, scoring=["accuracy", "f1_weighted", "f1_macro"], n_jobs=-1)
    m_f1 = float(c_scores["test_f1_macro"].mean())
    selection_rows.append({
        "Candidate Pipeline": c_name,
        "CV Accuracy": round(c_scores["test_accuracy"].mean(), 4),
        "CV Weighted F1": round(c_scores["test_f1_weighted"].mean(), 4),
        "CV Macro F1": round(m_f1, 4)
    })
    if m_f1 > best_cv_macro:
        best_cv_macro = m_f1
        best_overall_name = c_name
        best_model = c_pipe

oof_probs = cross_val_predict(best_model, X_train, y_train_class, cv=cv_strategy, method="predict_proba", n_jobs=-1)
oof_base_macro = f1_score(y_train_class, np.argmax(oof_probs, axis=1), average="macro")
best_oof_macro = oof_base_macro
best_mult = [1.0, 1.0, 1.0, 1.0]
for w1 in [0.9, 1.0, 1.1]:
    for w2 in [0.9, 1.0, 1.1]:
        for w3 in [0.9, 1.0, 1.1, 1.2]:
            mult = np.array([1.0, w1, w2, w3])
            score_m = f1_score(y_train_class, np.argmax(oof_probs * mult, axis=1), average="macro")
            if score_m > best_oof_macro + 0.0005:
                best_oof_macro = score_m
                best_mult = mult.tolist()

best_model.fit(X_train, y_train_class)
if hasattr(best_model, "named_steps") and "rfe" in best_model.named_steps:
    selected_rfe_features = [col for col, keep in zip(feature_cols, best_model.named_steps["rfe"].support_) if keep]
    print("Selected RFE Features:", selected_rfe_features)
print("OOF Threshold Multipliers:", best_mult, "| Validation Macro-F1:", round(best_oof_macro, 4))
print("Selected Best Overall Pipeline:", best_overall_name)
pd.DataFrame(selection_rows)

In [ ]:
original_rf_eval = ImbPipeline([("scaler", StandardScaler()), ("clf", RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42))])
original_rf_eval.fit(X_train, y_train_class)
tuned_rf_eval = search_rf_tuned.best_estimator_
tuned_rf_eval.fit(X_train, y_train_class)
smote_rf_eval = search_rf_smote.best_estimator_
smote_rf_eval.fit(X_train, y_train_class)

eval_models = [
    ("Original Random Forest", original_rf_eval),
    ("Tuned Random Forest", tuned_rf_eval),
    ("Best SMOTE Model", smote_rf_eval),
    ("Best Overall Model", best_model)
]

final_rows = []
for m_label, m_obj in eval_models:
    preds_m = m_obj.predict(X_test)
    probs_m = m_obj.predict_proba(X_test)
    final_rows.append({
        "Model": m_label,
        "Accuracy": round(accuracy_score(y_test_class, preds_m), 4),
        "Precision": round(precision_score(y_test_class, preds_m, average="weighted", zero_division=0), 4),
        "Recall": round(recall_score(y_test_class, preds_m, average="weighted", zero_division=0), 4),
        "F1-score": round(f1_score(y_test_class, preds_m, average="weighted", zero_division=0), 4),
        "Macro F1-score": round(f1_score(y_test_class, preds_m, average="macro", zero_division=0), 4),
        "ROC-AUC": round(roc_auc_score(y_test_class, probs_m, multi_class="ovr", average="weighted"), 4)
    })

final_comparison_df = pd.DataFrame(final_rows)
acc_gain = final_comparison_df.loc[3, "Accuracy"] - final_comparison_df.loc[0, "Accuracy"]
wf1_gain = final_comparison_df.loc[3, "F1-score"] - final_comparison_df.loc[0, "F1-score"]
mf1_gain = final_comparison_df.loc[3, "Macro F1-score"] - final_comparison_df.loc[0, "Macro F1-score"]
print("Accuracy Improvement:", round(acc_gain * 100, 2), "%")
print("Weighted F1 Improvement:", round(wf1_gain * 100, 2), "%")
print("Macro F1 Improvement:", round(mf1_gain * 100, 2), "%")
final_comparison_df

In [ ]:
best_test_preds = best_model.predict(X_test)
print("Detailed Classification Report — Best Overall Model:")
print(classification_report(y_test_class, best_test_preds, target_names=["Low", "Moderate", "High", "Severe"], digits=4))

cm_final = confusion_matrix(y_test_class, best_test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_final, annot=True, fmt="d", cmap="Blues", xticklabels=["Low", "Mod", "High", "Sev"], yticklabels=["Low", "Mod", "High", "Sev"])
plt.title("Confusion Matrix — Best Overall Model (Optimized Pipeline)")
plt.xlabel("Predicted Stress Level")
plt.ylabel("True Stress Level")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].bar(final_comparison_df["Model"], final_comparison_df["Accuracy"], color=["#64748b", "#0ea5e9", "#f59e0b", "#10b981"])
axes[0, 0].set_ylim(0.985, 1.001)
axes[0, 0].set_title("Model Accuracy Comparison")
axes[0, 0].set_ylabel("Accuracy")
axes[0, 0].tick_params(axis="x", rotation=18)

x_pos = np.arange(len(final_comparison_df))
axes[0, 1].bar(x_pos - 0.18, final_comparison_df["F1-score"], width=0.35, label="Weighted F1", color="#0f766e")
axes[0, 1].bar(x_pos + 0.18, final_comparison_df["Macro F1-score"], width=0.35, label="Macro F1", color="#d97706")
axes[0, 1].set_xticks(x_pos)
axes[0, 1].set_xticklabels(final_comparison_df["Model"], rotation=18)
axes[0, 1].set_ylim(0.980, 1.001)
axes[0, 1].set_title("Weighted vs Macro F1-Score Comparison")
axes[0, 1].set_ylabel("F1-Score")
axes[0, 1].legend()

imp_sorted = feature_importance_df.sort_values(by="Importance", ascending=True)
axes[1, 0].barh(imp_sorted["Feature"], imp_sorted["Importance"], color="forestgreen")
axes[1, 0].set_title("Feature Importance Ranking (14 Predictors)")
axes[1, 0].set_xlabel("Relative Importance")

y_test_bin = label_binarize(y_test_class, classes=[0, 1, 2, 3])
best_test_probs = best_model.predict_proba(X_test)
class_names_list = ["Low", "Moderate", "High", "Severe"]
roc_colors = ["#10b981", "#f59e0b", "#ea580c", "#dc2626"]
for idx_c in range(4):
    fpr_c, tpr_c, _ = roc_curve(y_test_bin[:, idx_c], best_test_probs[:, idx_c])
    roc_auc_c = auc(fpr_c, tpr_c)
    axes[1, 1].plot(fpr_c, tpr_c, color=roc_colors[idx_c], lw=2, label=class_names_list[idx_c] + " (AUC=" + str(round(roc_auc_c, 4)) + ")")
axes[1, 1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1, 1].set_title("Multi-Class One-vs-Rest ROC Curves")
axes[1, 1].set_xlabel("False Positive Rate")
axes[1, 1].set_ylabel("True Positive Rate")
axes[1, 1].legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
joblib.dump(scaler, "feature_scaler.pkl")
joblib.dump(best_model, "crop_stress_classifier.pkl")
joblib.dump(model_rf_reg, "irrigation_priority_regressor.pkl")
joblib.dump(model_dt_class, "crop_stress_decision_tree.pkl")
joblib.dump(model_logreg, "crop_stress_logistic_regression.pkl")
joblib.dump(model_dt_reg, "irrigation_decision_tree.pkl")
joblib.dump(model_linreg, "irrigation_linear_regression.pkl")

In [ ]:
df_districts = df[df["granularity"] == "district_baseline"].copy().reset_index(drop=True)
df_districts.head()

In [ ]:
X_732 = df_districts[feature_cols].copy()
X_732_scaled = pd.DataFrame(scaler.transform(X_732), columns=feature_cols)

In [ ]:
df_districts["Irrigation_Priority_Index"] = np.round(np.clip(model_rf_reg.predict(X_732_scaled), 0.0, 100.0), 1)
df_districts["Stress_Level_Code"] = best_model.predict(X_732)
stress_label_dict = {0: "Low", 1: "Moderate", 2: "High", 3: "Severe"}
df_districts["Crop_Water_Stress"] = df_districts["Stress_Level_Code"].map(stress_label_dict)

df_districts["Priority_Category"] = np.where(
    df_districts["Irrigation_Priority_Index"] >= 50.0, "High Priority",
    np.where(df_districts["Irrigation_Priority_Index"] >= 35.0, "Medium Priority", "Low Priority")
)

df_districts["Recommended_Water_Depth"] = np.where(
    df_districts["Irrigation_Priority_Index"] >= 50.0, "40–50 mm",
    np.where(df_districts["Irrigation_Priority_Index"] >= 35.0, "25–35 mm", "Routine / Nil")
)

df_ranked_732 = df_districts.sort_values(by="Irrigation_Priority_Index", ascending=False).reset_index(drop=True)
df_ranked_732["Rank"] = range(1, len(df_ranked_732) + 1)
df_ranked_732[["Rank", "district", "state", "Irrigation_Priority_Index", "Priority_Category", "Recommended_Water_Depth"]].head(10)

In [ ]:
plt.figure(figsize=(9, 5))
top10_districts = df_ranked_732.head(10)
sns.barplot(
    data=top10_districts,
    x="Irrigation_Priority_Index",
    y="district",
    hue="district",
    palette="Reds_r",
    legend=False
)
plt.title("Top 10 Indian Districts by Irrigation Priority")
plt.xlabel("Irrigation Priority Index (0 to 100)")
plt.ylabel("District")
plt.show()

In [ ]:
priority_summary = df_ranked_732["Priority_Category"].value_counts()
priority_summary

In [ ]:
clustering_features = [
    "temperature", "rainfall", "humidity", "effective_soil_wetness",
    "solar_radiation", "soil_ph", "soil_organic_carbon", "ndvi", "et0", "etc"
]
scaler_clustering = StandardScaler()
X_cluster_scaled = scaler_clustering.fit_transform(df_ranked_732[clustering_features])

kmeans_model = KMeans(n_clusters=4, random_state=42, n_init=10)
df_ranked_732["Cluster_ID"] = kmeans_model.fit_predict(X_cluster_scaled)

zone_names = {
    0: "Zone 0: Cool Mountain & High-Altitude",
    1: "Zone 1: Humid Coastal & Floodplain",
    2: "Zone 2: Semi-Arid & Deficit Core",
    3: "Zone 3: Wet Tropical & Heavy Rain"
}
df_ranked_732["Cluster_Zone"] = df_ranked_732["Cluster_ID"].map(zone_names)
df_ranked_732["Cluster_Zone"].value_counts()

In [ ]:
pca_model = PCA(n_components=2, random_state=42)
pca_coordinates = pca_model.fit_transform(X_cluster_scaled)
df_ranked_732["PCA_X"] = np.round(pca_coordinates[:, 0], 2)
df_ranked_732["PCA_Y"] = np.round(pca_coordinates[:, 1], 2)
pca_model.explained_variance_ratio_

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_ranked_732,
    x="PCA_X",
    y="PCA_Y",
    hue="Cluster_Zone",
    palette=["#3b82f6", "#10b981", "#f59e0b", "#ef4444"],
    alpha=0.8,
    s=45
)
plt.title("All-India 732 Districts Agro-Climatic PCA Projection")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
cluster_characterization = df_ranked_732.groupby("Cluster_Zone")[
    ["temperature", "rainfall", "humidity", "effective_soil_wetness", "Irrigation_Priority_Index"]
].mean().round(2)
cluster_characterization

In [ ]:
df_ranked_732.to_csv("all_india_732_districts_priority_and_clusters.csv", index=False)
df_ranked_732.shape